In [ ]:
# ==========================================
# STEP 1: GENERATE DIVERSE CPU WORKLOADS
# ==========================================

import pandas as pd
import numpy as np
import random

# ------------------------------------------
# CONFIGURATION
# ------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

NUM_WORKLOADS = 5000
MIN_PROCESSES = 4
MAX_PROCESSES = 15

rows = []

# ------------------------------------------
# WORKLOAD PATTERNS
# ------------------------------------------

workload_types = [
    "CPU-heavy",
    "IO-heavy",
    "Mixed",
    "Short-job-heavy",
    "Long-job-heavy",
    "Priority-sensitive",
    "Interactive",
    "Staggered-arrival",
    "Similar-arrival",
    "Mixed-priority"
]

# ------------------------------------------
# GENERATE WORKLOADS
# ------------------------------------------

for workload_id in range(1, NUM_WORKLOADS + 1):

    workload_type = random.choice(workload_types)

    n_processes = random.randint(
        MIN_PROCESSES,
        MAX_PROCESSES
    )

    # --------------------------------------
    # Generate processes
    # --------------------------------------

    for process_num in range(1, n_processes + 1):

        process_id = f"P{process_num}"

        # ==================================
        # ARRIVAL TIME
        # ==================================

        if workload_type == "Similar-arrival":

            arrival = random.choice([0, 0, 1, 1, 2, 3])

        elif workload_type == "Staggered-arrival":

            arrival = min(
                20,
                int((process_num - 1) * 20 / n_processes)
                + random.randint(0, 2)
            )

        elif workload_type == "Interactive":

            arrival = random.randint(0, 20)

        elif workload_type == "Priority-sensitive":

            # Several processes arrive close together
            # so priority ordering actually matters.
            arrival = random.randint(0, 3)

        else:

            arrival = random.randint(0, 20)

        # ==================================
        # BURST TIME + PROCESS TYPE
        # ==================================

        if workload_type == "CPU-heavy":

            burst = random.randint(15, 30)
            process_type = "CPU-bound"

        elif workload_type == "IO-heavy":

            burst = random.randint(1, 8)
            process_type = "IO-bound"

        elif workload_type == "Short-job-heavy":

            burst = random.randint(1, 6)

            process_type = random.choice([
                "IO-bound",
                "IO-bound",
                "CPU-bound"
            ])

        elif workload_type == "Long-job-heavy":

            burst = random.randint(20, 30)

            process_type = random.choice([
                "CPU-bound",
                "CPU-bound",
                "IO-bound"
            ])

        elif workload_type == "Mixed":

            burst = random.randint(1, 30)

            process_type = random.choice([
                "CPU-bound",
                "IO-bound"
            ])

        elif workload_type == "Interactive":

            burst = random.randint(1, 8)

            process_type = "IO-bound"

        elif workload_type == "Staggered-arrival":

            burst = random.randint(3, 25)

            process_type = random.choice([
                "CPU-bound",
                "IO-bound"
            ])

        elif workload_type == "Similar-arrival":

            burst = random.randint(1, 30)

            process_type = random.choice([
                "CPU-bound",
                "IO-bound"
            ])

        elif workload_type == "Priority-sensitive":

            # ----------------------------------
            # IMPORTANT:
            #
            # Create a real priority-sensitive
            # workload.
            #
            # Priority and burst time are NOT
            # simply random.
            #
            # Some high-priority processes are
            # short, while some are long.
            # This creates genuine competition
            # between Priority and SJF.
            # ----------------------------------

            pattern = process_num % 4

            if pattern == 1:

                burst = random.randint(2, 5)
                priority = 1
                process_type = "IO-bound"

            elif pattern == 2:

                burst = random.randint(12, 20)
                priority = 1
                process_type = "CPU-bound"

            elif pattern == 3:

                burst = random.randint(3, 8)
                priority = random.choice([3, 4])
                process_type = "IO-bound"

            else:

                burst = random.randint(18, 30)
                priority = random.choice([4, 5])
                process_type = "CPU-bound"

        elif workload_type == "Mixed-priority":

            burst = random.randint(1, 30)

            process_type = random.choice([
                "CPU-bound",
                "IO-bound"
            ])

        else:

            burst = random.randint(1, 30)

            process_type = random.choice([
                "CPU-bound",
                "IO-bound"
            ])

        # ==================================
        # PRIORITY
        # ==================================

        if workload_type == "Priority-sensitive":

            # Priority already assigned above.
            pass

        elif workload_type == "Mixed-priority":

            priority = random.choice([
                1, 1, 2, 2, 3, 4, 5
            ])

        elif workload_type == "Interactive":

            # Some interactive processes receive
            # higher priority.
            priority = random.choice([
                1, 2, 2, 3, 4, 5
            ])

        else:

            priority = random.randint(1, 5)

        # ==================================
        # SAVE PROCESS
        # ==================================

        rows.append({

            "Workload_ID": workload_id,

            "Process_ID": process_id,

            "Arrival_Time": arrival,

            "Burst_Time": burst,

            "Priority": priority,

            "Process_Type": process_type

        })


# ------------------------------------------
# CREATE DATAFRAME
# ------------------------------------------

raw_dataset = pd.DataFrame(rows)


# ------------------------------------------
# VALIDATION
# ------------------------------------------

print("==========================================")
print("DATASET GENERATION COMPLETE")
print("==========================================")

print(
    "Number of workloads:",
    raw_dataset["Workload_ID"].nunique()
)

print(
    "Total process records:",
    len(raw_dataset)
)

print(
    "Minimum processes in a workload:",
    raw_dataset.groupby("Workload_ID").size().min()
)

print(
    "Maximum processes in a workload:",
    raw_dataset.groupby("Workload_ID").size().max()
)


# ------------------------------------------
# CHECK VALUES
# ------------------------------------------

print("\nMissing values:")
print(raw_dataset.isnull().sum())


print("\nPriority distribution:")
print(
    raw_dataset["Priority"]
    .value_counts()
    .sort_index()
)


print("\nProcess type distribution:")
print(
    raw_dataset["Process_Type"]
    .value_counts()
)


print("\nFirst 10 rows:")
display(raw_dataset.head(10))


print("\nDataset shape:")
print(raw_dataset.shape)

DATASET GENERATION COMPLETE
Number of workloads: 5000
Total process records: 47580
Minimum processes in a workload: 4
Maximum processes in a workload: 15

Missing values:
Workload_ID     0
Process_ID      0
Arrival_Time    0
Burst_Time      0
Priority        0
Process_Type    0
dtype: int64

Priority distribution:
Priority
1    11657
2     9535
3     8524
4     9287
5     8577
Name: count, dtype: int64

Process type distribution:
Process_Type
IO-bound     26795
CPU-bound    20785
Name: count, dtype: int64

First 10 rows:


,Workload_ID,Process_ID,Arrival_Time,Burst_Time,Priority,Process_Type
0,1,P1,8,4,2,IO-bound
1,1,P2,4,2,5,IO-bound
2,1,P3,2,7,1,IO-bound
3,1,P4,0,2,2,IO-bound
4,2,P1,19,1,2,CPU-bound
5,2,P2,20,6,4,CPU-bound
6,2,P3,7,4,3,CPU-bound
7,2,P4,0,2,4,CPU-bound
8,2,P5,10,3,2,IO-bound
9,2,P6,10,1,4,IO-bound



Dataset shape:
(47580, 6)


In [ ]:
raw_dataset.to_csv(
    "cpu_process_dataset.csv",
    index=False
)

print("Dataset saved as cpu_process_dataset.csv")

Dataset saved as cpu_process_dataset.csv


In [ ]:
# ==========================================
# STEP 2: CPU SCHEDULING ALGORITHMS
# ==========================================

def fcfs(processes):
    """
    First Come First Serve Scheduling
    """

    processes = sorted(
        processes,
        key=lambda x: (x["arrival"], x["pid"])
    )

    current_time = 0
    result = []

    for p in processes:

        # CPU waits if process has not arrived
        if current_time < p["arrival"]:
            current_time = p["arrival"]

        start_time = current_time

        current_time += p["burst"]

        completion_time = current_time

        result.append({
            "pid": p["pid"],
            "arrival": p["arrival"],
            "burst": p["burst"],
            "priority": p["priority"],
            "start": start_time,
            "completion": completion_time
        })

    return result


def sjf(processes):
    """
    Shortest Job First - Non-preemptive
    """

    remaining = processes.copy()

    current_time = 0
    result = []

    while remaining:

        # Processes that have already arrived
        available = [
            p for p in remaining
            if p["arrival"] <= current_time
        ]

        # If no process has arrived yet
        if not available:

            current_time = min(
                p["arrival"] for p in remaining
            )

            available = [
                p for p in remaining
                if p["arrival"] <= current_time
            ]

        # Select shortest burst
        selected = min(
            available,
            key=lambda x: (x["burst"], x["arrival"], x["pid"])
        )

        start_time = current_time

        current_time += selected["burst"]

        completion_time = current_time

        result.append({
            "pid": selected["pid"],
            "arrival": selected["arrival"],
            "burst": selected["burst"],
            "priority": selected["priority"],
            "start": start_time,
            "completion": completion_time
        })

        remaining.remove(selected)

    return result


def priority_scheduling(processes):
    """
    Non-preemptive Priority Scheduling
    Priority 1 = highest priority
    """

    remaining = processes.copy()

    current_time = 0
    result = []

    while remaining:

        available = [
            p for p in remaining
            if p["arrival"] <= current_time
        ]

        if not available:

            current_time = min(
                p["arrival"] for p in remaining
            )

            available = [
                p for p in remaining
                if p["arrival"] <= current_time
            ]

        # Smaller priority number = higher priority
        selected = min(
            available,
            key=lambda x: (x["priority"], x["arrival"], x["pid"])
        )

        start_time = current_time

        current_time += selected["burst"]

        completion_time = current_time

        result.append({
            "pid": selected["pid"],
            "arrival": selected["arrival"],
            "burst": selected["burst"],
            "priority": selected["priority"],
            "start": start_time,
            "completion": completion_time
        })

        remaining.remove(selected)

    return result


def round_robin(processes, quantum=4):
    """
    Round Robin Scheduling
    Time Quantum = 4
    """

    processes = sorted(
        processes,
        key=lambda x: (x["arrival"], x["pid"])
    )

    # Make copies
    remaining = {
        p["pid"]: p["burst"]
        for p in processes
    }

    first_start = {
        p["pid"]: None
        for p in processes
    }

    completion = {
        p["pid"]: None
        for p in processes
    }

    arrival = {
        p["pid"]: p["arrival"]
        for p in processes
    }

    burst = {
        p["pid"]: p["burst"]
        for p in processes
    }

    priority = {
        p["pid"]: p["priority"]
        for p in processes
    }

    ready_queue = []

    current_time = 0
    index = 0

    context_switches = 0
    previous_pid = None

    while index < len(processes) or ready_queue:

        # If queue is empty, jump to next arrival
        if not ready_queue:

            if index < len(processes):

                if current_time < processes[index]["arrival"]:
                    current_time = processes[index]["arrival"]

        # Add newly arrived processes
        while (
            index < len(processes)
            and processes[index]["arrival"] <= current_time
        ):

            ready_queue.append(
                processes[index]["pid"]
            )

            index += 1

        if not ready_queue:
            continue

        pid = ready_queue.pop(0)

        # Context switch
        if previous_pid is not None and previous_pid != pid:
            context_switches += 1

        # First response time
        if first_start[pid] is None:
            first_start[pid] = current_time

        execution_time = min(
            quantum,
            remaining[pid]
        )

        current_time += execution_time

        remaining[pid] -= execution_time

        # Add newly arrived processes
        while (
            index < len(processes)
            and processes[index]["arrival"] <= current_time
        ):

            ready_queue.append(
                processes[index]["pid"]
            )

            index += 1

        # Process finished
        if remaining[pid] == 0:

            completion[pid] = current_time

        else:

            # Put it back into queue
            ready_queue.append(pid)

        previous_pid = pid

    result = []

    for p in processes:

        result.append({
            "pid": p["pid"],
            "arrival": arrival[p["pid"]],
            "burst": burst[p["pid"]],
            "priority": priority[p["pid"]],
            "start": first_start[p["pid"]],
            "completion": completion[p["pid"]]
        })

    return result, context_switches


print("All four scheduling algorithms created successfully!")

All four scheduling algorithms created successfully!


In [ ]:

# ==========================================
# STEP 3: CORRECTED PERFORMANCE METRICS
# ==========================================

import numpy as np

def calculate_metrics(processes, schedule, algorithm,
                      context_switches=None):

    if not processes or not schedule:
        raise ValueError("Processes and schedule must not be empty.")

    schedule_dict = {p["pid"]: p for p in schedule}

    waiting_times = []
    turnaround_times = []
    response_times = []

    for p in processes:
        pid = p["pid"]
        item = schedule_dict[pid]

        arrival = p["arrival"]
        burst = p["burst"]
        start = item["start"]
        completion = item["completion"]

        turnaround = completion - arrival
        waiting = turnaround - burst
        response = start - arrival

        if waiting < 0 or response < 0 or turnaround < burst:
            raise ValueError(f"Invalid scheduling times for {pid}")

        waiting_times.append(waiting)
        turnaround_times.append(turnaround)
        response_times.append(response)

    first_arrival = min(p["arrival"] for p in processes)
    last_completion = max(p["completion"] for p in schedule)
    total_time = last_completion - first_arrival
    total_burst = sum(p["burst"] for p in processes)

    throughput = len(processes) / total_time if total_time > 0 else 0

    cpu_utilization = (
        min(100.0, total_burst / total_time * 100)
        if total_time > 0 else 0
    )

    if context_switches is None:
        ordered = sorted(schedule, key=lambda p: p["start"])
        context_switches = sum(
            ordered[i]["pid"] != ordered[i - 1]["pid"]
            for i in range(1, len(ordered))
        )

    return {
        "Algorithm": algorithm,
        "Waiting_Time": round(float(np.mean(waiting_times)), 2),
        "Turnaround_Time": round(float(np.mean(turnaround_times)), 2),
        "Response_Time": round(float(np.mean(response_times)), 2),
        "Throughput": round(float(throughput), 4),
        "Context_Switches": int(context_switches),
        "CPU_Utilization": round(float(cpu_utilization), 2)
    }

print("Corrected performance metrics function created successfully!")


Corrected performance metrics function created successfully!


In [ ]:

# ==========================================
# TEST ONE WORKLOAD
# ==========================================

# Select Workload 1
workload_1 = raw_dataset[
    raw_dataset["Workload_ID"] == 1
]

# Convert dataframe into process dictionaries
processes = []

for _, row in workload_1.iterrows():
    processes.append({
        "pid": row["Process_ID"],
        "arrival": int(row["Arrival_Time"]),
        "burst": int(row["Burst_Time"]),
        "priority": int(row["Priority"])
    })


# Run FCFS
fcfs_result = fcfs(processes)

# Run SJF
sjf_result = sjf(processes)

# Run Priority
priority_result = priority_scheduling(processes)

# Run Round Robin
rr_result, rr_context_switches = round_robin(
    processes,
    quantum=4
)


# Calculate metrics
# FCFS, SJF and Priority count process transitions
# automatically inside calculate_metrics().

fcfs_metrics = calculate_metrics(
    processes, fcfs_result, "FCFS"
)

sjf_metrics = calculate_metrics(
    processes, sjf_result, "SJF"
)

priority_metrics = calculate_metrics(
    processes, priority_result, "Priority"
)

# Round Robin supplies its own transition count.
rr_metrics = calculate_metrics(
    processes,
    rr_result,
    "Round Robin",
    context_switches=rr_context_switches
)


# Display results
results = pd.DataFrame([
    fcfs_metrics,
    sjf_metrics,
    priority_metrics,
    rr_metrics
])

print("Scheduling comparison for Workload 1:")
display(results)


Scheduling comparison for Workload 1:


,Algorithm,Waiting_Time,Turnaround_Time,Response_Time,Throughput,Context_Switches,CPU_Utilization
0,FCFS,2.00,5.75,2.00,0.2667,3,100.0
1,SJF,2.00,5.75,2.00,0.2667,3,100.0
2,Priority,2.50,6.25,2.50,0.2667,3,100.0
3,Round Robin,1.75,5.50,1.25,0.2667,4,100.0


In [ ]:

# ============================================================
# STEP 4: RUN ALL WORKLOADS AND CREATE ML DATASET
# ============================================================

all_results = []

for workload_id in raw_dataset["Workload_ID"].unique():

    # Get one workload
    workload_data = raw_dataset[
        raw_dataset["Workload_ID"] == workload_id
    ]

    if workload_data.empty:
        continue

    # Convert to process format
    processes = []

    for _, row in workload_data.iterrows():
        processes.append({
            "pid": row["Process_ID"],
            "arrival": int(row["Arrival_Time"]),
            "burst": int(row["Burst_Time"]),
            "priority": int(row["Priority"])
        })

    # Run all four scheduling algorithms
    fcfs_result = fcfs(processes)
    sjf_result = sjf(processes)
    priority_result = priority_scheduling(processes)

    rr_result, rr_context_switches = round_robin(
        processes,
        quantum=4
    )

    # Calculate metrics
    fcfs_metrics = calculate_metrics(
        processes, fcfs_result, "FCFS"
    )

    sjf_metrics = calculate_metrics(
        processes, sjf_result, "SJF"
    )

    priority_metrics = calculate_metrics(
        processes, priority_result, "Priority"
    )

    rr_metrics = calculate_metrics(
        processes,
        rr_result,
        "Round Robin",
        context_switches=rr_context_switches
    )

    # Store results
    for metrics in [
        fcfs_metrics,
        sjf_metrics,
        priority_metrics,
        rr_metrics
    ]:
        metrics["Workload_ID"] = workload_id
        all_results.append(metrics)


# Convert results into a DataFrame
all_metrics = pd.DataFrame(all_results)

# Rearrange columns
all_metrics = all_metrics[
    [
        "Workload_ID",
        "Algorithm",
        "Waiting_Time",
        "Turnaround_Time",
        "Response_Time",
        "Throughput",
        "Context_Switches",
        "CPU_Utilization"
    ]
]

print("All workloads evaluated successfully!")
print(
    "Number of workloads:",
    all_metrics["Workload_ID"].nunique()
)
print(
    "Number of evaluation records:",
    len(all_metrics)
)

display(all_metrics.head(12))


All workloads evaluated successfully!
Number of workloads: 5000
Number of evaluation records: 20000


,Workload_ID,Algorithm,Waiting_Time,Turnaround_Time,Response_Time,Throughput,Context_Switches,CPU_Utilization
0,1,FCFS,2.00,5.75,2.00,0.2667,3,100.0
1,1,SJF,2.00,5.75,2.00,0.2667,3,100.0
2,1,Priority,2.50,6.25,2.50,0.2667,3,100.0
3,1,Round Robin,1.75,5.50,1.25,0.2667,4,100.0
4,2,FCFS,6.67,9.67,6.67,0.3333,11,100.0
5,2,SJF,4.58,7.58,4.58,0.3333,11,100.0
6,2,Priority,7.25,10.25,7.25,0.3333,11,100.0
7,2,Round Robin,7.25,10.25,6.17,0.3333,14,100.0
8,3,FCFS,6.43,11.00,6.43,0.2188,6,100.0
9,3,SJF,5.29,9.86,5.29,0.2188,6,100.0


In [ ]:

# ==========================================
# STEP 4C: DETERMINE BEST SCHEDULING STRATEGY
# ==========================================

scored_metrics = all_metrics.copy()

# Metrics where lower values are better
lower_metrics = [
    "Waiting_Time",
    "Turnaround_Time",
    "Response_Time",
    "Context_Switches"
]

# Metrics where higher values are better
higher_metrics = [
    "Throughput",
    "CPU_Utilization"
]

scored_groups = []

for workload_id, group in scored_metrics.groupby("Workload_ID"):

    group = group.copy()

    # Start with a neutral score for every metric
    for metric in lower_metrics + higher_metrics:
        group[f"{metric}_Score"] = 0.5

    # Normalize only metrics that differ across algorithms
    for metric in lower_metrics:

        values = group[metric]
        min_value = values.min()
        max_value = values.max()

        if max_value > min_value:
            group[f"{metric}_Score"] = (
                (max_value - values) /
                (max_value - min_value)
            )

    for metric in higher_metrics:

        values = group[metric]
        min_value = values.min()
        max_value = values.max()

        if max_value > min_value:
            group[f"{metric}_Score"] = (
                (values - min_value) /
                (max_value - min_value)
            )

    # Weighted performance score
    group["Overall_Score"] = (
        0.25 * group["Waiting_Time_Score"]
        + 0.20 * group["Turnaround_Time_Score"]
        + 0.20 * group["Response_Time_Score"]
        + 0.15 * group["Throughput_Score"]
        + 0.10 * group["Context_Switches_Score"]
        + 0.10 * group["CPU_Utilization_Score"]
    )

    # Select the highest-scoring strategy.
    # If scores tie, use waiting time, turnaround time,
    # response time, then context switches as tie-breakers.
    ranked = group.sort_values(
        by=[
            "Overall_Score",
            "Waiting_Time",
            "Turnaround_Time",
            "Response_Time",
            "Context_Switches",
            "Algorithm"
        ],
        ascending=[
            False, True, True, True, True, True
        ],
        kind="stable"
    )

    best_strategy = ranked.iloc[0]["Algorithm"]

    # Store one consistent label for this workload
    group["Best_Strategy"] = best_strategy

    scored_groups.append(group)

# Combine the results
all_metrics = pd.concat(
    scored_groups,
    ignore_index=True
)

# Display strategy distribution
strategy_distribution = (
    all_metrics
    .groupby("Workload_ID")["Best_Strategy"]
    .first()
    .value_counts()
)

print("Best scheduling strategies calculated!")
print("\nStrategy distribution:")
print(strategy_distribution)

print(
    "\nTotal workloads:",
    all_metrics["Workload_ID"].nunique()
)

# Display detailed scores for Workload 1
print("\nExample: Workload 1")

display(
    all_metrics[
        all_metrics["Workload_ID"] == 1
    ][
        [
            "Workload_ID",
            "Algorithm",
            "Waiting_Time",
            "Turnaround_Time",
            "Response_Time",
            "Throughput",
            "Context_Switches",
            "CPU_Utilization",
            "Overall_Score",
            "Best_Strategy"
        ]
    ]
)


Best scheduling strategies calculated!

Strategy distribution:
Best_Strategy
SJF            4562
FCFS            243
Priority        136
Round Robin      59
Name: count, dtype: int64

Total workloads: 5000

Example: Workload 1


,Workload_ID,Algorithm,Waiting_Time,Turnaround_Time,Response_Time,Throughput,Context_Switches,CPU_Utilization,Overall_Score,Best_Strategy
0,1,FCFS,2.00,5.75,2.00,0.2667,3,100.0,0.605,Round Robin
1,1,SJF,2.00,5.75,2.00,0.2667,3,100.0,0.605,Round Robin
2,1,Priority,2.50,6.25,2.50,0.2667,3,100.0,0.225,Round Robin
3,1,Round Robin,1.75,5.50,1.25,0.2667,4,100.0,0.775,Round Robin


In [ ]:
# ==========================================
# STEP 4D: CREATE FINAL ML DATASET
# ==========================================

ml_rows = []

for workload_id, workload_data in raw_dataset.groupby("Workload_ID"):

    # ------------------------------------------
    # WORKLOAD FEATURES
    # ------------------------------------------

    num_processes = len(workload_data)

    avg_burst = workload_data["Burst_Time"].mean()
    min_burst = workload_data["Burst_Time"].min()
    max_burst = workload_data["Burst_Time"].max()
    burst_std = workload_data["Burst_Time"].std()

    if pd.isna(burst_std):
        burst_std = 0

    burst_range = max_burst - min_burst

    avg_arrival = workload_data["Arrival_Time"].mean()
    min_arrival = workload_data["Arrival_Time"].min()
    max_arrival = workload_data["Arrival_Time"].max()

    arrival_range = max_arrival - min_arrival
    arrival_std = workload_data["Arrival_Time"].std()

    if pd.isna(arrival_std):
        arrival_std = 0

    avg_priority = workload_data["Priority"].mean()
    min_priority = workload_data["Priority"].min()
    max_priority = workload_data["Priority"].max()

    priority_std = workload_data["Priority"].std()

    if pd.isna(priority_std):
        priority_std = 0

    # ------------------------------------------
    # CPU / I/O CHARACTERISTICS
    # ------------------------------------------

    cpu_ratio = (
        workload_data["Process_Type"]
        .eq("CPU-bound")
        .mean()
    )

    io_ratio = (
        workload_data["Process_Type"]
        .eq("IO-bound")
        .mean()
    )

    # ------------------------------------------
    # GET RESULTS FOR THIS WORKLOAD
    # ------------------------------------------

    workload_results = all_metrics[
        all_metrics["Workload_ID"] == workload_id
    ].copy()

    # ------------------------------------------
    # GET BEST STRATEGY
    # ------------------------------------------

    best_strategy = (
        workload_results["Best_Strategy"]
        .iloc[0]
    )

    # ------------------------------------------
    # GET PERFORMANCE RESULTS
    # ------------------------------------------

    performance = {}

    for _, row in workload_results.iterrows():

        algorithm = row["Algorithm"]

        if algorithm == "FCFS":
            prefix = "FCFS"

        elif algorithm == "SJF":
            prefix = "SJF"

        elif algorithm == "Priority":
            prefix = "Priority"

        elif algorithm == "Round Robin":
            prefix = "RR"

        else:
            continue

        performance[
            f"{prefix}_Average_Waiting_Time"
        ] = row["Waiting_Time"]

        performance[
            f"{prefix}_Average_Turnaround_Time"
        ] = row["Turnaround_Time"]

        performance[
            f"{prefix}_Average_Response_Time"
        ] = row["Response_Time"]

        performance[
            f"{prefix}_Throughput"
        ] = row["Throughput"]

        performance[
            f"{prefix}_Context_Switches"
        ] = row["Context_Switches"]

        performance[
            f"{prefix}_CPU_Utilization"
        ] = row["CPU_Utilization"]

    # ------------------------------------------
    # CREATE ONE ROW FOR THIS WORKLOAD
    # ------------------------------------------

    ml_rows.append({

        "Workload_ID": workload_id,

        "Number_of_Processes": num_processes,

        "Average_Burst_Time": avg_burst,
        "Minimum_Burst_Time": min_burst,
        "Maximum_Burst_Time": max_burst,
        "Burst_Time_Std": burst_std,
        "Burst_Time_Range": burst_range,

        "Average_Arrival_Time": avg_arrival,
        "Minimum_Arrival_Time": min_arrival,
        "Maximum_Arrival_Time": max_arrival,
        "Arrival_Time_Range": arrival_range,
        "Arrival_Time_Std": arrival_std,

        "Average_Priority": avg_priority,
        "Minimum_Priority": min_priority,
        "Maximum_Priority": max_priority,
        "Priority_Std": priority_std,

        "CPU_Bound_Ratio": cpu_ratio,
        "IO_Bound_Ratio": io_ratio,

        **performance,

        "Best_Strategy": best_strategy
    })


# ==========================================
# CREATE DATAFRAME
# ==========================================

ml_dataset = pd.DataFrame(ml_rows)

ml_dataset = ml_dataset.sort_values(
    "Workload_ID"
).reset_index(drop=True)


# ==========================================
# DISPLAY RESULTS
# ==========================================

print("==========================================")
print("FINAL ML DATASET CREATED")
print("==========================================")

print("Number of workloads:", len(ml_dataset))
print("Number of columns:", len(ml_dataset.columns))

print("\nDataset shape:")
print(ml_dataset.shape)

print("\nFirst 10 rows:")
display(ml_dataset.head(10))

print("\nBest Strategy Distribution:")
print(
    ml_dataset["Best_Strategy"].value_counts()
)

print("\nMissing values:")
print(
    ml_dataset.isnull().sum().sum()
)

FINAL ML DATASET CREATED
Number of workloads: 5000
Number of columns: 43

Dataset shape:
(5000, 43)

First 10 rows:


,Workload_ID,Number_of_Processes,Average_Burst_Time,Minimum_Burst_Time,Maximum_Burst_Time,Burst_Time_Std,Burst_Time_Range,Average_Arrival_Time,Minimum_Arrival_Time,Maximum_Arrival_Time,...,Priority_Throughput,Priority_Context_Switches,Priority_CPU_Utilization,RR_Average_Waiting_Time,RR_Average_Turnaround_Time,RR_Average_Response_Time,RR_Throughput,RR_Context_Switches,RR_CPU_Utilization,Best_Strategy
0,1,4,3.750000,2,7,2.362908,5,3.500000,0,8,...,0.2667,3,100.00,1.75,5.50,1.25,0.2667,4,100.00,Round Robin
1,2,12,3.000000,1,6,1.906925,5,10.250000,0,20,...,0.3333,11,100.00,7.25,10.25,6.17,0.3333,14,100.00,SJF
2,3,7,4.571429,2,7,1.902379,5,10.714286,3,20,...,0.2188,6,100.00,8.57,13.14,4.86,0.2188,10,100.00,SJF
3,4,14,2.928571,1,6,1.730464,5,12.142857,2,20,...,0.3333,13,97.62,11.36,14.29,8.50,0.3333,16,97.62,SJF
4,5,8,18.875000,6,30,8.509449,24,1.750000,0,3,...,0.0530,7,100.00,92.12,111.00,12.25,0.0530,41,100.00,SJF
5,6,8,12.750000,2,30,10.067628,28,1.125000,0,3,...,0.0784,7,100.00,44.00,56.75,11.62,0.0784,26,100.00,SJF
6,7,12,13.583333,4,25,6.082140,21,9.750000,1,18,...,0.0736,11,100.00,91.00,104.58,18.58,0.0736,44,100.00,SJF
7,8,7,2.142857,1,5,1.676163,4,7.714286,0,13,...,0.3684,6,78.95,0.86,3.00,0.71,0.3684,7,78.95,FCFS
8,9,10,20.300000,16,30,5.437524,14,8.100000,1,16,...,0.0493,9,100.00,134.50,154.80,16.10,0.0493,53,100.00,SJF
9,10,10,4.200000,2,7,1.751190,5,11.200000,2,20,...,0.2381,9,100.00,11.00,15.20,8.00,0.2381,13,100.00,SJF



Best Strategy Distribution:
Best_Strategy
SJF            4562
FCFS            243
Priority        136
Round Robin      59
Name: count, dtype: int64

Missing values:
0


In [ ]:
# ==========================================
# STEP 5: FINAL DATASET VALIDATION
# ==========================================

print("==========================================")
print("FINAL DATASET VALIDATION")
print("==========================================")

# ------------------------------------------
# 1. Dataset size
# ------------------------------------------

print("\n1. Dataset Size")
print("Rows:", len(ml_dataset))
print("Columns:", len(ml_dataset.columns))


# ------------------------------------------
# 2. Number of workloads
# ------------------------------------------

print("\n2. Number of Workloads")
print(
    "Unique workloads:",
    ml_dataset["Workload_ID"].nunique()
)


# ------------------------------------------
# 3. Missing values
# ------------------------------------------

print("\n3. Missing Values")

total_missing = ml_dataset.isnull().sum().sum()

print("Total missing values:", total_missing)


# ------------------------------------------
# 4. Duplicate workload IDs
# ------------------------------------------

print("\n4. Duplicate Workload IDs")

duplicate_ids = ml_dataset["Workload_ID"].duplicated().sum()

print("Duplicate IDs:", duplicate_ids)


# ------------------------------------------
# 5. Best Strategy distribution
# ------------------------------------------

print("\n5. Best Strategy Distribution")

print(
    ml_dataset["Best_Strategy"]
    .value_counts()
)


# ------------------------------------------
# 6. Percentage distribution
# ------------------------------------------

print("\n6. Strategy Percentages")

print(
    (ml_dataset["Best_Strategy"]
     .value_counts(normalize=True) * 100)
     .round(2)
)


# ------------------------------------------
# 7. Check process count
# ------------------------------------------

print("\n7. Number of Processes")

print(
    "Minimum:",
    ml_dataset["Number_of_Processes"].min()
)

print(
    "Maximum:",
    ml_dataset["Number_of_Processes"].max()
)


# ------------------------------------------
# 8. Check CPU/I/O ratios
# ------------------------------------------

print("\n8. CPU/I/O Ratio Check")

print(
    "Minimum CPU ratio:",
    ml_dataset["CPU_Bound_Ratio"].min()
)

print(
    "Maximum CPU ratio:",
    ml_dataset["CPU_Bound_Ratio"].max()
)

print(
    "Minimum I/O ratio:",
    ml_dataset["IO_Bound_Ratio"].min()
)

print(
    "Maximum I/O ratio:",
    ml_dataset["IO_Bound_Ratio"].max()
)


# ------------------------------------------
# 9. Check target classes
# ------------------------------------------

print("\n9. Number of Strategy Classes")

print(
    ml_dataset["Best_Strategy"].nunique()
)


# ------------------------------------------
# FINAL STATUS
# ------------------------------------------

if (
    len(ml_dataset) == 5000
    and
    ml_dataset["Workload_ID"].nunique() == 5000
    and
    total_missing == 0
    and
    duplicate_ids == 0
    and
    ml_dataset["Best_Strategy"].nunique() >= 2
):

    print("\n==========================================")
    print("DATASET VALIDATION PASSED!")
    print("==========================================")

else:

    print("\n==========================================")
    print("DATASET VALIDATION NEEDS ATTENTION")
    print("==========================================")

FINAL DATASET VALIDATION

1. Dataset Size
Rows: 5000
Columns: 43

2. Number of Workloads
Unique workloads: 5000

3. Missing Values
Total missing values: 0

4. Duplicate Workload IDs
Duplicate IDs: 0

5. Best Strategy Distribution
Best_Strategy
SJF            4562
FCFS            243
Priority        136
Round Robin      59
Name: count, dtype: int64

6. Strategy Percentages
Best_Strategy
SJF            91.24
FCFS            4.86
Priority        2.72
Round Robin     1.18
Name: proportion, dtype: float64

7. Number of Processes
Minimum: 4
Maximum: 15

8. CPU/I/O Ratio Check
Minimum CPU ratio: 0.0
Maximum CPU ratio: 1.0
Minimum I/O ratio: 0.0
Maximum I/O ratio: 1.0

9. Number of Strategy Classes
4

DATASET VALIDATION PASSED!


In [ ]:
# ==========================================
# STEP 6: SAVE FINAL ML DATASET
# ==========================================

file_name = "scheduling_ml_dataset.csv"

ml_dataset.to_csv(
    file_name,
    index=False
)

print("==========================================")
print("FINAL DATASET SAVED")
print("==========================================")

print("File name:", file_name)
print("Rows:", len(ml_dataset))
print("Columns:", len(ml_dataset.columns))

FINAL DATASET SAVED
File name: scheduling_ml_dataset.csv
Rows: 5000
Columns: 43


In [ ]:
# ==========================================
# RANDOM FOREST - STEP 1
# PREPARE DATA
# ==========================================

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import pandas as pd
import numpy as np


# ==========================================
# INPUT FEATURES
# ==========================================

feature_columns = [

    "Number_of_Processes",

    "Average_Burst_Time",
    "Minimum_Burst_Time",
    "Maximum_Burst_Time",
    "Burst_Time_Std",
    "Burst_Time_Range",

    "Average_Arrival_Time",
    "Minimum_Arrival_Time",
    "Maximum_Arrival_Time",
    "Arrival_Time_Range",
    "Arrival_Time_Std",

    "Average_Priority",
    "Minimum_Priority",
    "Maximum_Priority",
    "Priority_Std",

    "CPU_Bound_Ratio",
    "IO_Bound_Ratio"
]


# ==========================================
# CREATE INPUT (X) AND TARGET (y)
# ==========================================

X = ml_dataset[feature_columns]

y = ml_dataset["Best_Strategy"]


# ==========================================
# DISPLAY INFORMATION
# ==========================================

print("==========================================")
print("RANDOM FOREST - DATA PREPARATION")
print("==========================================")

print("\nNumber of samples:", len(X))

print("Number of input features:", len(feature_columns))

print("\nX shape:")
print(X.shape)

print("\ny shape:")
print(y.shape)

print("\nInput features:")

for feature in feature_columns:
    print("-", feature)

print("\nTarget variable:")
print("Best_Strategy")

print("\nTarget classes:")
print(y.unique())

print("\nTarget distribution:")
print(y.value_counts())

print("\nMissing values in X:")
print(X.isnull().sum().sum())

print("\nMissing values in y:")
print(y.isnull().sum())

RANDOM FOREST - DATA PREPARATION

Number of samples: 5000
Number of input features: 17

X shape:
(5000, 17)

y shape:
(5000,)

Input features:
- Number_of_Processes
- Average_Burst_Time
- Minimum_Burst_Time
- Maximum_Burst_Time
- Burst_Time_Std
- Burst_Time_Range
- Average_Arrival_Time
- Minimum_Arrival_Time
- Maximum_Arrival_Time
- Arrival_Time_Range
- Arrival_Time_Std
- Average_Priority
- Minimum_Priority
- Maximum_Priority
- Priority_Std
- CPU_Bound_Ratio
- IO_Bound_Ratio

Target variable:
Best_Strategy

Target classes:
['Round Robin' 'SJF' 'FCFS' 'Priority']

Target distribution:
Best_Strategy
SJF            4562
FCFS            243
Priority        136
Round Robin      59
Name: count, dtype: int64

Missing values in X:
0

Missing values in y:
0


In [ ]:
# ==========================================
# RANDOM FOREST - STEP 2
# TRAIN / TEST SPLIT
# ==========================================

from sklearn.model_selection import train_test_split

# ------------------------------------------
# Split the dataset
# ------------------------------------------
# 80% -> Training
# 20% -> Testing
#
# stratify=y keeps the same class proportions
# in both training and testing datasets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ==========================================
# DISPLAY SPLIT INFORMATION
# ==========================================

print("==========================================")
print("TRAIN / TEST SPLIT COMPLETE")
print("==========================================")

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nX_train shape:")
print(X_train.shape)

print("\nX_test shape:")
print(X_test.shape)

print("\ny_train shape:")
print(y_train.shape)

print("\ny_test shape:")
print(y_test.shape)


# ==========================================
# CHECK CLASS DISTRIBUTION
# ==========================================

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

TRAIN / TEST SPLIT COMPLETE

Training samples: 4000
Testing samples: 1000

X_train shape:
(4000, 17)

X_test shape:
(1000, 17)

y_train shape:
(4000,)

y_test shape:
(1000,)

Training target distribution:
Best_Strategy
SJF            3650
FCFS            194
Priority        109
Round Robin      47
Name: count, dtype: int64

Testing target distribution:
Best_Strategy
SJF            912
FCFS            49
Priority        27
Round Robin     12
Name: count, dtype: int64


In [ ]:

# ==========================================
# STEP 7: TRAIN AND COMPARE FOUR ML MODELS
# ==========================================

%pip -q install xgboost

import numpy as np
import pandas as pd

from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from xgboost import XGBClassifier

# ------------------------------------------
# 1. Balance TRAINING data only
# ------------------------------------------

training_data = X_train.copy()
training_data["Best_Strategy"] = y_train.values

largest_class = training_data["Best_Strategy"].value_counts().max()
balanced_parts = []

for strategy, group in training_data.groupby("Best_Strategy"):
    balanced_group = resample(
        group,
        replace=True,
        n_samples=largest_class,
        random_state=42
    )
    balanced_parts.append(balanced_group)

balanced_data = pd.concat(
    balanced_parts, ignore_index=True
).sample(frac=1, random_state=42).reset_index(drop=True)

X_train_balanced = balanced_data[feature_columns]
y_train_balanced = balanced_data["Best_Strategy"]

print("Original training size:", len(X_train))
print("Balanced training size:", len(X_train_balanced))
print("\nBalanced class distribution:")
print(y_train_balanced.value_counts())

# ------------------------------------------
# 2. Encode labels for XGBoost
# ------------------------------------------

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train_balanced)

# ------------------------------------------
# 3. Define the four models
# ------------------------------------------

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.1,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ),

    "Neural Network": make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=400,
        early_stopping=False,
        random_state=42
    )
)

}

# ------------------------------------------
# 4. Train and evaluate
# ------------------------------------------

trained_models = {}
comparison_results = {}

for name, model in models.items():

    print(f"\nTraining {name}...")

    if name == "XGBoost":
        model.fit(X_train_balanced, y_train_encoded)

        encoded_predictions = model.predict(X_test).astype(int)
        predictions = label_encoder.inverse_transform(
            encoded_predictions
        )
    else:
        model.fit(X_train_balanced, y_train_balanced)
        predictions = model.predict(X_test)

    trained_models[name] = model

    comparison_results[name] = {
        "Accuracy (%)": round(
            accuracy_score(y_test, predictions) * 100, 2
        ),
        "Balanced Accuracy (%)": round(
            balanced_accuracy_score(y_test, predictions) * 100, 2
        ),
        "Macro F1": round(
            f1_score(
                y_test, predictions,
                average="macro",
                zero_division=0
            ), 4
        )
    }

    print(f"{name} completed.")

# ------------------------------------------
# 5. Compare all models
# ------------------------------------------

comparison_df = pd.DataFrame(
    comparison_results
).T

comparison_df.index.name = "Model"

comparison_df = comparison_df.sort_values(
    "Macro F1", ascending=False
)

print("\n========== MODEL COMPARISON ==========")
display(comparison_df)

# ------------------------------------------
# 6. Detailed results for the best model
# ------------------------------------------

best_model_name = comparison_df.index[0]
best_model = trained_models[best_model_name]

if best_model_name == "XGBoost":
    best_predictions = label_encoder.inverse_transform(
        best_model.predict(X_test).astype(int)
    )
else:
    best_predictions = best_model.predict(X_test)

print("\nBest model by Macro F1:", best_model_name)

print("\nClassification report:")
print(classification_report(
    y_test,
    best_predictions,
    labels=["FCFS", "SJF", "Priority", "Round Robin"],
    zero_division=0
))

print("\nConfusion matrix:")
display(pd.DataFrame(
    confusion_matrix(
        y_test,
        best_predictions,
        labels=["FCFS", "SJF", "Priority", "Round Robin"]
    ),
    index=["Actual FCFS", "Actual SJF",
           "Actual Priority", "Actual Round Robin"],
    columns=["Pred FCFS", "Pred SJF",
             "Pred Priority", "Pred Round Robin"]
))


Original training size: 4000
Balanced training size: 14600

Balanced class distribution:
Best_Strategy
FCFS           3650
Round Robin    3650
Priority       3650
SJF            3650
Name: count, dtype: int64

Training Random Forest...
Random Forest completed.

Training Decision Tree...
Decision Tree completed.

Training XGBoost...
XGBoost completed.

Training Neural Network...
Neural Network completed.

========== MODEL COMPARISON ==========


,Accuracy (%),Balanced Accuracy (%),Macro F1
Model,,,
XGBoost,87.9,46.28,0.4291
Neural Network,88.0,42.61,0.4008
Random Forest,90.5,35.30,0.3571
Decision Tree,86.1,33.73,0.3279



Best model by Macro F1: XGBoost

Classification report:
              precision    recall  f1-score   support

        FCFS       0.30      0.45      0.36        49
         SJF       0.97      0.93      0.95       912
    Priority       0.14      0.22      0.17        27
 Round Robin       0.21      0.25      0.23        12

    accuracy                           0.88      1000
   macro avg       0.41      0.46      0.43      1000
weighted avg       0.91      0.88      0.89      1000


Confusion matrix:


,Pred FCFS,Pred SJF,Pred Priority,Pred Round Robin
Actual FCFS,22,11,12,4
Actual SJF,38,848,20,6
Actual Priority,9,11,6,1
Actual Round Robin,4,1,4,3


In [ ]:

# ==========================================
# STEP 8: ENSEMBLE MODEL
# RANDOM FOREST + XGBOOST
# ==========================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("==========================================")
print("ENSEMBLE MODEL: RANDOM FOREST + XGBOOST")
print("==========================================")

# Get the two already-trained models
rf_model = trained_models["Random Forest"]
xgb_model = trained_models["XGBoost"]

# Get class labels in a common order
class_labels = label_encoder.classes_

# 1. Random Forest probabilities
rf_probabilities = rf_model.predict_proba(X_test)

# Reorder Random Forest columns to match label_encoder.classes_
rf_class_order = list(rf_model.classes_)
rf_probabilities = rf_probabilities[
    :, [rf_class_order.index(label) for label in class_labels]
]

# 2. XGBoost probabilities
xgb_probabilities = xgb_model.predict_proba(X_test)

# 3. Combine both models using soft voting
ensemble_probabilities = (
    rf_probabilities + xgb_probabilities
) / 2

# 4. Get final predictions
ensemble_predictions = class_labels[
    np.argmax(ensemble_probabilities, axis=1)
]

# 5. Calculate performance metrics
results = {
    "Random Forest": {
        "Accuracy (%)": accuracy_score(
            y_test, rf_model.predict(X_test)
        ) * 100,
        "Balanced Accuracy (%)": balanced_accuracy_score(
            y_test, rf_model.predict(X_test)
        ) * 100,
        "Macro F1": f1_score(
            y_test, rf_model.predict(X_test),
            average="macro", zero_division=0
        )
    },

    "XGBoost": {
        "Accuracy (%)": accuracy_score(
            y_test,
            label_encoder.inverse_transform(
                xgb_model.predict(X_test).astype(int)
            )
        ) * 100,
        "Balanced Accuracy (%)": balanced_accuracy_score(
            y_test,
            label_encoder.inverse_transform(
                xgb_model.predict(X_test).astype(int)
            )
        ) * 100,
        "Macro F1": f1_score(
            y_test,
            label_encoder.inverse_transform(
                xgb_model.predict(X_test).astype(int)
            ),
            average="macro", zero_division=0
        )
    },

    "Ensemble (RF + XGBoost)": {
        "Accuracy (%)": accuracy_score(
            y_test, ensemble_predictions
        ) * 100,
        "Balanced Accuracy (%)": balanced_accuracy_score(
            y_test, ensemble_predictions
        ) * 100,
        "Macro F1": f1_score(
            y_test, ensemble_predictions,
            average="macro", zero_division=0
        )
    }
}

# 6. Display comparison table
comparison = pd.DataFrame(results).T

comparison = comparison.round({
    "Accuracy (%)": 2,
    "Balanced Accuracy (%)": 2,
    "Macro F1": 4
})

print("\n========== FINAL MODEL RESULTS ==========")
display(comparison)

# 7. Display detailed ensemble result
print("\n========== ENSEMBLE CLASSIFICATION REPORT ==========")

print(classification_report(
    y_test,
    ensemble_predictions,
    labels=class_labels,
    zero_division=0
))

print("\n========== ENSEMBLE CONFUSION MATRIX ==========")

display(pd.DataFrame(
    confusion_matrix(
        y_test,
        ensemble_predictions,
        labels=class_labels
    ),
    index=[f"Actual {x}" for x in class_labels],
    columns=[f"Predicted {x}" for x in class_labels]
))

# 8. Save ensemble information for later use
ensemble_model = {
    "random_forest": rf_model,
    "xgboost": xgb_model,
    "label_encoder": label_encoder,
    "feature_columns": feature_columns
}

import joblib

joblib.dump(
    ensemble_model,
    "cpu_scheduling_ensemble_model.pkl"
)

print("\nEnsemble model saved successfully!")
print("File: cpu_scheduling_ensemble_model.pkl")


ENSEMBLE MODEL: RANDOM FOREST + XGBOOST

========== FINAL MODEL RESULTS ==========


,Accuracy (%),Balanced Accuracy (%),Macro F1
Random Forest,90.5,35.30,0.3571
XGBoost,87.9,46.28,0.4291
Ensemble (RF + XGBoost),88.7,42.64,0.4066



========== ENSEMBLE CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

        FCFS       0.31      0.45      0.37        49
    Priority       0.14      0.15      0.15        27
 Round Robin       0.15      0.17      0.16        12
         SJF       0.97      0.94      0.95       912

    accuracy                           0.89      1000
   macro avg       0.39      0.43      0.41      1000
weighted avg       0.90      0.89      0.89      1000


========== ENSEMBLE CONFUSION MATRIX ==========


,Predicted FCFS,Predicted Priority,Predicted Round Robin,Predicted SJF
Actual FCFS,22,10,2,15
Actual Priority,8,4,2,13
Actual Round Robin,6,3,2,1
Actual SJF,35,11,7,859



Ensemble model saved successfully!
File: cpu_scheduling_ensemble_model.pkl


In [ ]:

# ==========================================
# ENSEMBLE MODEL: XGBOOST + NEURAL NETWORK
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)
import numpy as np
import pandas as pd
import joblib

# Get the trained models
xgb_model = trained_models["XGBoost"]
nn_model = trained_models["Neural Network"]

# Common class order
class_labels = label_encoder.classes_

# XGBoost probabilities
xgb_prob = xgb_model.predict_proba(X_test)

# Convert XGBoost encoded classes to actual labels
xgb_labels = label_encoder.inverse_transform(
    xgb_model.classes_.astype(int)
)

# Reorder XGBoost probability columns
xgb_prob = xgb_prob[:, [
    list(xgb_labels).index(label) for label in class_labels
]]

# Neural Network probabilities
nn_prob = nn_model.predict_proba(X_test)

# Reorder Neural Network probability columns
nn_prob = nn_prob[:, [
    list(nn_model.classes_).index(label) for label in class_labels
]]

# Soft Voting: Average both model probabilities
ensemble_prob = (xgb_prob + nn_prob) / 2

# Final predictions
ensemble_pred = class_labels[np.argmax(ensemble_prob, axis=1)]

# Calculate results
results = {
    "XGBoost + Neural Network": {
        "Accuracy (%)": round(
            accuracy_score(y_test, ensemble_pred) * 100, 2
        ),
        "Balanced Accuracy (%)": round(
            balanced_accuracy_score(y_test, ensemble_pred) * 100, 2
        ),
        "Macro F1": round(
            f1_score(y_test, ensemble_pred,
                     average="macro", zero_division=0), 4
        )
    }
}

print("====== XGBOOST + NEURAL NETWORK RESULTS ======")
display(pd.DataFrame(results).T)

print("\n====== CLASSIFICATION REPORT ======")
print(classification_report(
    y_test,
    ensemble_pred,
    labels=class_labels,
    zero_division=0
))

# Save this ensemble
joblib.dump(
    {
        "xgboost": xgb_model,
        "neural_network": nn_model,
        "label_encoder": label_encoder,
        "feature_columns": feature_columns
    },
    "xgboost_neural_network_ensemble.pkl"
)

print("\nEnsemble model saved successfully!")


====== XGBOOST + NEURAL NETWORK RESULTS ======


,Accuracy (%),Balanced Accuracy (%),Macro F1
XGBoost + Neural Network,88.4,44.65,0.4157



====== CLASSIFICATION REPORT ======
              precision    recall  f1-score   support

        FCFS       0.37      0.51      0.43        49
    Priority       0.16      0.26      0.20        27
 Round Robin       0.08      0.08      0.08        12
         SJF       0.97      0.93      0.95       912

    accuracy                           0.88      1000
   macro avg       0.40      0.45      0.42      1000
weighted avg       0.91      0.88      0.90      1000


Ensemble model saved successfully!


In [ ]:

# ==========================================
# STEP 8: SAVE THE BEST ML MODEL
# ==========================================

import joblib

# Save the best model selected by Macro F1
joblib.dump(
    {
        "model_name": best_model_name,
        "model": best_model,
        "label_encoder": label_encoder,
        "feature_columns": feature_columns
    },
    "best_cpu_scheduling_model.pkl"
)

print("Best model saved successfully!")
print("Model:", best_model_name)
print("Selection metric: Macro F1")
print("Macro F1:", comparison_df.loc[best_model_name, "Macro F1"])
print("File: best_cpu_scheduling_model.pkl")


Best model saved successfully!
Model: XGBoost
Selection metric: Macro F1
Macro F1: 0.4291
File: best_cpu_scheduling_model.pkl


In [ ]:

# ==========================================
# SAVE FINAL ENSEMBLE MODEL
# RANDOM FOREST + XGBOOST
# ==========================================

import joblib

rf_model = trained_models["Random Forest"]
xgb_model = trained_models["XGBoost"]

ensemble_model = {
    "model_name": "Random Forest + XGBoost",
    "random_forest": rf_model,
    "xgboost": xgb_model,
    "label_encoder": label_encoder,
    "feature_columns": feature_columns
}

joblib.dump(
    ensemble_model,
    "cpu_scheduling_ensemble_model.pkl"
)

print("Final ensemble model saved successfully!")
print("Selected models: Random Forest + XGBoost")
print("Method: Soft Voting")
print("File: cpu_scheduling_ensemble_model.pkl")


Final ensemble model saved successfully!
Selected models: Random Forest + XGBoost
Method: Soft Voting
File: cpu_scheduling_ensemble_model.pkl


In [ ]:

# ==========================================
# STEP 9: RANDOM FOREST + XGBOOST PREDICTOR
# ==========================================

def predict_from_processes(processes):

    if not processes:
        raise ValueError("Enter at least one process.")

    df = pd.DataFrame(processes)

    # ------------------------------------------
    # 1. Check required columns
    # ------------------------------------------

    required_columns = [
        "Arrival_Time",
        "Burst_Time",
        "Priority",
        "Process_Type"
    ]

    for column in required_columns:
        if column not in df.columns:
            raise ValueError(f"Missing column: {column}")

    # ------------------------------------------
    # 2. Validate numerical inputs
    # ------------------------------------------

    for column in ["Arrival_Time", "Burst_Time", "Priority"]:
        df[column] = pd.to_numeric(df[column], errors="raise")

    # Remove extra spaces and tabs
    df["Process_Type"] = (
        df["Process_Type"].astype(str).str.strip()
    )

    if not np.isfinite(
        df[["Arrival_Time", "Burst_Time", "Priority"]].to_numpy()
    ).all():
        raise ValueError("Process values must be finite numbers.")

    if (df["Arrival_Time"] < 0).any():
        raise ValueError("Arrival time cannot be negative.")

    if (df["Burst_Time"] <= 0).any():
        raise ValueError("Burst time must be greater than zero.")

    if not df["Process_Type"].isin(
        ["CPU-bound", "IO-bound"]
    ).all():
        raise ValueError(
            "Process type must be CPU-bound or IO-bound."
        )

    # ------------------------------------------
    # 3. Extract workload features
    # ------------------------------------------

    workload_features = pd.DataFrame([{

        "Number_of_Processes": len(df),

        "Average_Burst_Time": df["Burst_Time"].mean(),
        "Minimum_Burst_Time": df["Burst_Time"].min(),
        "Maximum_Burst_Time": df["Burst_Time"].max(),
        "Burst_Time_Std": df["Burst_Time"].std(),
        "Burst_Time_Range": (
            df["Burst_Time"].max() - df["Burst_Time"].min()
        ),

        "Average_Arrival_Time": df["Arrival_Time"].mean(),
        "Minimum_Arrival_Time": df["Arrival_Time"].min(),
        "Maximum_Arrival_Time": df["Arrival_Time"].max(),
        "Arrival_Time_Range": (
            df["Arrival_Time"].max() - df["Arrival_Time"].min()
        ),
        "Arrival_Time_Std": df["Arrival_Time"].std(),

        "Average_Priority": df["Priority"].mean(),
        "Minimum_Priority": df["Priority"].min(),
        "Maximum_Priority": df["Priority"].max(),
        "Priority_Std": df["Priority"].std(),

        "CPU_Bound_Ratio": (
            df["Process_Type"].eq("CPU-bound").mean()
        ),

        "IO_Bound_Ratio": (
            df["Process_Type"].eq("IO-bound").mean()
        )

    }]).fillna(0)

    # ------------------------------------------
    # 4. Arrange features in training order
    # ------------------------------------------

    X_new = workload_features[feature_columns]

    # ------------------------------------------
    # 5. Get trained models
    # ------------------------------------------

    rf_model = trained_models["Random Forest"]
    xgb_model = trained_models["XGBoost"]

    class_labels = label_encoder.classes_

    # ------------------------------------------
    # 6. Random Forest probabilities
    # ------------------------------------------

    rf_prob = rf_model.predict_proba(X_new)

    rf_labels = list(rf_model.classes_)

    rf_prob = rf_prob[:, [
        rf_labels.index(label)
        for label in class_labels
    ]]

    # ------------------------------------------
    # 7. XGBoost probabilities
    # ------------------------------------------

    xgb_prob = xgb_model.predict_proba(X_new)

    xgb_labels = label_encoder.inverse_transform(
        xgb_model.classes_.astype(int)
    )

    xgb_labels = list(xgb_labels)

    xgb_prob = xgb_prob[:, [
        xgb_labels.index(label)
        for label in class_labels
    ]]

    # ------------------------------------------
    # 8. Soft Voting
    # ------------------------------------------

    ensemble_prob = (rf_prob + xgb_prob) / 2

    prediction = class_labels[
        np.argmax(ensemble_prob, axis=1)
    ][0]

    # ------------------------------------------
    # 9. Display prediction probabilities
    # ------------------------------------------

    probability_table = pd.DataFrame({

        "Scheduling_Strategy": class_labels,

        "Probability (%)": ensemble_prob[0] * 100

    }).sort_values(
        "Probability (%)",
        ascending=False
    ).reset_index(drop=True)

    probability_table["Probability (%)"] = (
        probability_table["Probability (%)"].round(2)
    )

    # ------------------------------------------
    # 10. Display final recommendation
    # ------------------------------------------

    print("==========================================")
    print("RANDOM FOREST + XGBOOST PREDICTION")
    print("==========================================")

    print("Number of processes:", len(df))

    print("\nRecommended Scheduling Strategy:")
    print("***", prediction, "***")

    print("\nPrediction probabilities:")
    display(probability_table)

    return prediction, workload_features


print(
    "Random Forest + XGBoost workload predictor "
    "defined successfully!"
)


Random Forest + XGBoost workload predictor defined successfully!


In [ ]:
# ==========================================
# STEP 10
# TEST NEW WORKLOAD
# ==========================================

new_workload = [

    {
        "Arrival_Time": 0,
        "Burst_Time": 8,
        "Priority": 2,
        "Process_Type": "CPU-bound"
    },

    {
        "Arrival_Time": 1,
        "Burst_Time": 4,
        "Priority": 1,
        "Process_Type": "IO-bound"
    },

    {
        "Arrival_Time": 2,
        "Burst_Time": 12,
        "Priority": 3,
        "Process_Type": "CPU-bound"
    },

    {
        "Arrival_Time": 3,
        "Burst_Time": 3,
        "Priority": 2,
        "Process_Type": "IO-bound"
    },

    {
        "Arrival_Time": 4,
        "Burst_Time": 10,
        "Priority": 1,
        "Process_Type": "CPU-bound"
    },

    {
        "Arrival_Time": 5,
        "Burst_Time": 5,
        "Priority": 4,
        "Process_Type": "IO-bound"
    }
]


new_workload_prediction, workload_features = (
    predict_from_processes(new_workload)
)


print("\nWorkload features used by the model:")
display(workload_features)

RANDOM FOREST + XGBOOST PREDICTION
Number of processes: 6

Recommended Scheduling Strategy:
*** SJF ***

Prediction probabilities:


,Scheduling_Strategy,Probability (%)
0,SJF,77.07
1,FCFS,11.52
2,Priority,9.95
3,Round Robin,1.45



Workload features used by the model:


,Number_of_Processes,Average_Burst_Time,Minimum_Burst_Time,Maximum_Burst_Time,Burst_Time_Std,Burst_Time_Range,Average_Arrival_Time,Minimum_Arrival_Time,Maximum_Arrival_Time,Arrival_Time_Range,Arrival_Time_Std,Average_Priority,Minimum_Priority,Maximum_Priority,Priority_Std,CPU_Bound_Ratio,IO_Bound_Ratio
0,6,7.0,3,12,3.577709,9,2.5,0,5,5,1.870829,2.166667,1,4,1.169045,0.5,0.5


In [ ]:
# ==========================================
# STEP 11
# FINAL CPU SCHEDULING ML DEMO
# ==========================================

print("==========================================")
print(" CPU SCHEDULING STRATEGY PREDICTOR")
print("==========================================")

print("\nEnter the workload information.")
print("Type 'done' when you have entered all processes.\n")


processes = []

process_number = 1

while True:

    print(f"\nProcess P{process_number}")

    arrival = input("Arrival Time: ")

    if arrival.lower() == "done":
        break

    burst = input("Burst Time: ")
    priority = input("Priority: ")
    process_type = input(
        "Process Type (CPU-bound / IO-bound): "
    )

    processes.append({

        "Arrival_Time": float(arrival),

        "Burst_Time": float(burst),

        "Priority": float(priority),

        "Process_Type": process_type
    })

    process_number += 1


# ------------------------------------------
# Check workload
# ------------------------------------------

if len(processes) == 0:

    print("\nNo processes entered.")

else:

    print("\n==========================================")
    print("WORKLOAD ENTERED")
    print("==========================================")

    workload_df = pd.DataFrame(processes)

    display(workload_df)


    # --------------------------------------
    # Predict
    # --------------------------------------

    prediction, workload_features = (
        predict_from_processes(processes)
    )


    # --------------------------------------
    # Final recommendation
    # --------------------------------------

    print("\n==========================================")
    print("FINAL RECOMMENDATION")
    print("==========================================")

    print(
        f"\nRecommended CPU Scheduling Strategy:"
    )

    print(
        f"        >>> {prediction} <<<"
    )
    print("The recommendation was generated")
    print("using the Random Forest + XGBoost Ensemble.")
    print("Combination method: Soft Voting")

 CPU SCHEDULING STRATEGY PREDICTOR

Enter the workload information.
Type 'done' when you have entered all processes.


Process P1
Arrival Time: 0
Burst Time: 5
Priority: 2
Process Type (CPU-bound / IO-bound): CPU-bound

Process P2
Arrival Time: 1
Burst Time: 3
Priority: 1
Process Type (CPU-bound / IO-bound): IO-bound

Process P3
Arrival Time: 2
Burst Time: 8
Priority: 3
Process Type (CPU-bound / IO-bound): CPU-bound

Process P4
Arrival Time: 3
Burst Time: 2
Priority: 2
Process Type (CPU-bound / IO-bound): IO-bound

Process P5
Arrival Time: done

WORKLOAD ENTERED


,Arrival_Time,Burst_Time,Priority,Process_Type
0,0.0,5.0,2.0,CPU-bound
1,1.0,3.0,1.0,IO-bound
2,2.0,8.0,3.0,CPU-bound
3,3.0,2.0,2.0,IO-bound


RANDOM FOREST + XGBOOST PREDICTION
Number of processes: 4

Recommended Scheduling Strategy:
*** SJF ***

Prediction probabilities:


,Scheduling_Strategy,Probability (%)
0,SJF,37.63
1,Round Robin,23.07
2,Priority,20.38
3,FCFS,18.92



FINAL RECOMMENDATION

Recommended CPU Scheduling Strategy:
        >>> SJF <<<
The recommendation was generated
using the Random Forest + XGBoost Ensemble.
Combination method: Soft Voting


In [ ]:

# ==========================================
# STEP 12
# COMPARE ALL SCHEDULING ALGORITHMS
# ==========================================

print("==========================================")
print("SCHEDULING PERFORMANCE COMPARISON")
print("==========================================")

# Convert entered workload into scheduling format

simulation_processes = []

for i, process in enumerate(processes):

    simulation_processes.append({
        "pid": f"P{i + 1}",
        "arrival": int(process["Arrival_Time"]),
        "burst": int(process["Burst_Time"]),
        "priority": int(process["Priority"])
    })


# ------------------------------------------
# Run all four algorithms
# ------------------------------------------

fcfs_result = fcfs(simulation_processes)

sjf_result = sjf(simulation_processes)

priority_result = priority_scheduling(
    simulation_processes
)

rr_result, rr_context_switches = round_robin(
    simulation_processes,
    quantum=4
)


# ------------------------------------------
# Calculate performance metrics
# ------------------------------------------

fcfs_metrics = calculate_metrics(
    simulation_processes,
    fcfs_result,
    "FCFS"
)

sjf_metrics = calculate_metrics(
    simulation_processes,
    sjf_result,
    "SJF"
)

priority_metrics = calculate_metrics(
    simulation_processes,
    priority_result,
    "Priority"
)

rr_metrics = calculate_metrics(
    simulation_processes,
    rr_result,
    "Round Robin",
    rr_context_switches
)


# ------------------------------------------
# Combine and display results
# ------------------------------------------

comparison = pd.DataFrame([
    fcfs_metrics,
    sjf_metrics,
    priority_metrics,
    rr_metrics
])

print("\nPerformance of all algorithms:")

display(comparison)


# ------------------------------------------
# Show Ensemble ML recommendation
# ------------------------------------------

print("\n==========================================")
print("ML ENSEMBLE RECOMMENDATION")
print("==========================================")

print(
    f"Random Forest + XGBoost prediction: "
    f">>> {prediction} <<<"
)

print("Combination Method: Soft Voting")


SCHEDULING PERFORMANCE COMPARISON

Performance of all algorithms:


,Algorithm,Waiting_Time,Turnaround_Time,Response_Time,Throughput,Context_Switches,CPU_Utilization
0,FCFS,5.75,10.25,5.75,0.2222,3,100.0
1,SJF,4.00,8.50,4.00,0.2222,3,100.0
2,Priority,4.25,8.75,4.25,0.2222,3,100.0
3,Round Robin,7.00,11.50,4.00,0.2222,5,100.0



ML ENSEMBLE RECOMMENDATION
Random Forest + XGBoost prediction: >>> SJF <<<
Combination Method: Soft Voting


In [ ]:

# ==========================================================
# STEP 13: COMPLETE PROJECT RESULTS SUMMARY
# ==========================================================

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

print("\n")
print("=" * 65)
print("       ML-BASED CPU SCHEDULING STRATEGY PREDICTION")
print("                 FINAL PROJECT RESULTS")
print("=" * 65)


# ----------------------------------------------------------
# 1. PROJECT OVERVIEW
# ----------------------------------------------------------

print("\n1. PROJECT OVERVIEW")
print("-" * 65)

print("Project Title: ML-Based CPU Scheduling Strategy Prediction")

print("Objective:")
print("To use Machine Learning to predict the most suitable")
print("CPU scheduling algorithm based on process workload.")

print("\nScheduling Algorithms:")
print("1. First Come First Serve (FCFS)")
print("2. Shortest Job First (SJF)")
print("3. Priority Scheduling")
print("4. Round Robin")


# ----------------------------------------------------------
# 2. DATASET INFORMATION
# ----------------------------------------------------------

print("\n2. DATASET INFORMATION")
print("-" * 65)

print("Number of Workloads       : 5,000")
print("Total Process Records     : 47,580")
print("Features Used             : 17")
print("Dataset Columns           : 43")
print("Training Samples          : 4,000")
print("Testing Samples           : 1,000")

class_distribution = pd.DataFrame({
    "Scheduling Strategy": [
        "SJF", "FCFS", "Priority", "Round Robin"
    ],
    "Number of Workloads": [
        4562, 243, 136, 59
    ],
    "Percentage": [
        91.24, 4.86, 2.72, 1.18
    ]
})

print("\nClass Distribution:")
display(class_distribution)


# ----------------------------------------------------------
# 3. MACHINE LEARNING MODEL COMPARISON
# ----------------------------------------------------------

print("\n3. MACHINE LEARNING MODEL COMPARISON")
print("-" * 65)

model_results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Decision Tree",
        "XGBoost",
        "Neural Network"
    ],
    "Accuracy (%)": [
        90.5, 86.1, 87.9, 88.0
    ],
    "Balanced Accuracy (%)": [
        35.30, 33.73, 46.28, 42.61
    ],
    "Macro F1": [
        0.3571, 0.3279, 0.4291, 0.4008
    ]
})

display(model_results)


# ----------------------------------------------------------
# 4. FINAL ENSEMBLE MODEL
# ----------------------------------------------------------

print("\n4. FINAL ENSEMBLE MODEL")
print("-" * 65)

print("Selected Models : Random Forest + XGBoost")
print("Combination     : Soft Voting")
print("Ensemble Accuracy             : 88.7%")
print("Ensemble Balanced Accuracy    : 42.64%")
print("Ensemble Macro F1             : 0.4066")

print("\nReason for Selection:")
print("The two models were combined to satisfy the ensemble")
print("approach and compare their joint prediction performance.")


# ----------------------------------------------------------
# 5. ENTERED WORKLOAD
# ----------------------------------------------------------

print("\n5. ENTERED WORKLOAD")
print("-" * 65)

workload_display = pd.DataFrame(processes)
display(workload_display)

print("Total Processes:", len(processes))


# ----------------------------------------------------------
# 6. FINAL ML PREDICTION
# ----------------------------------------------------------

print("\n6. FINAL ML PREDICTION")
print("-" * 65)

print("Selected Ensemble: Random Forest + XGBoost")
print("Combination Method: Soft Voting")

print("\nRecommended Scheduling Strategy:")
print("       >>>", prediction, "<<<")


# ----------------------------------------------------------
# 7. SCHEDULING ALGORITHM PERFORMANCE
# ----------------------------------------------------------

print("\n7. SCHEDULING ALGORITHM PERFORMANCE")
print("-" * 65)

display(comparison)


# ----------------------------------------------------------
# 8. BEST PERFORMING ALGORITHM FOR THIS WORKLOAD
# ----------------------------------------------------------

print("\n8. BEST PERFORMING ALGORITHM FOR THIS WORKLOAD")
print("-" * 65)

best_algorithm = comparison.loc[
    comparison["Waiting_Time"].idxmin()
]

print("Algorithm with lowest average waiting time:",
      best_algorithm["Algorithm"])

print("Average Waiting Time:",
      best_algorithm["Waiting_Time"])

print("Average Turnaround Time:",
      best_algorithm["Turnaround_Time"])


# ----------------------------------------------------------
# 9. FINAL CONCLUSION
# ----------------------------------------------------------

print("\n9. FINAL PROJECT CONCLUSION")
print("-" * 65)

print("The project successfully trained multiple Machine")
print("Learning models to predict CPU scheduling strategies.")

print("Random Forest and XGBoost were combined using Soft Voting.")

print("The final ensemble predicted:",
      prediction)

print("The four scheduling algorithms were also executed")
print("and their performance metrics were compared.")

print("\n")
print("=" * 65)
print("              PROJECT EXECUTION COMPLETED")
print("=" * 65)




       ML-BASED CPU SCHEDULING STRATEGY PREDICTION
                 FINAL PROJECT RESULTS

1. PROJECT OVERVIEW
-----------------------------------------------------------------
Project Title: ML-Based CPU Scheduling Strategy Prediction
Objective:
To use Machine Learning to predict the most suitable
CPU scheduling algorithm based on process workload.

Scheduling Algorithms:
1. First Come First Serve (FCFS)
2. Shortest Job First (SJF)
3. Priority Scheduling
4. Round Robin

2. DATASET INFORMATION
-----------------------------------------------------------------
Number of Workloads       : 5,000
Total Process Records     : 47,580
Features Used             : 17
Dataset Columns           : 43
Training Samples          : 4,000
Testing Samples           : 1,000

Class Distribution:


,Scheduling Strategy,Number of Workloads,Percentage
0,SJF,4562,91.24
1,FCFS,243,4.86
2,Priority,136,2.72
3,Round Robin,59,1.18



3. MACHINE LEARNING MODEL COMPARISON
-----------------------------------------------------------------


,Model,Accuracy (%),Balanced Accuracy (%),Macro F1
0,Random Forest,90.5,35.30,0.3571
1,Decision Tree,86.1,33.73,0.3279
2,XGBoost,87.9,46.28,0.4291
3,Neural Network,88.0,42.61,0.4008



4. FINAL ENSEMBLE MODEL
-----------------------------------------------------------------
Selected Models : Random Forest + XGBoost
Combination     : Soft Voting
Ensemble Accuracy             : 88.7%
Ensemble Balanced Accuracy    : 42.64%
Ensemble Macro F1             : 0.4066

Reason for Selection:
The two models were combined to satisfy the ensemble
approach and compare their joint prediction performance.

5. ENTERED WORKLOAD
-----------------------------------------------------------------


,Arrival_Time,Burst_Time,Priority,Process_Type
0,0.0,5.0,2.0,CPU-bound
1,1.0,3.0,1.0,IO-bound
2,2.0,8.0,3.0,CPU-bound
3,3.0,2.0,2.0,IO-bound


Total Processes: 4

6. FINAL ML PREDICTION
-----------------------------------------------------------------
Selected Ensemble: Random Forest + XGBoost
Combination Method: Soft Voting

Recommended Scheduling Strategy:
       >>> SJF <<<

7. SCHEDULING ALGORITHM PERFORMANCE
-----------------------------------------------------------------


,Algorithm,Waiting_Time,Turnaround_Time,Response_Time,Throughput,Context_Switches,CPU_Utilization
0,FCFS,5.75,10.25,5.75,0.2222,3,100.0
1,SJF,4.00,8.50,4.00,0.2222,3,100.0
2,Priority,4.25,8.75,4.25,0.2222,3,100.0
3,Round Robin,7.00,11.50,4.00,0.2222,5,100.0



8. BEST PERFORMING ALGORITHM FOR THIS WORKLOAD
-----------------------------------------------------------------
Algorithm with lowest average waiting time: SJF
Average Waiting Time: 4.0
Average Turnaround Time: 8.5

9. FINAL PROJECT CONCLUSION
-----------------------------------------------------------------
The project successfully trained multiple Machine
Learning models to predict CPU scheduling strategies.
Random Forest and XGBoost were combined using Soft Voting.
The final ensemble predicted: SJF
The four scheduling algorithms were also executed
and their performance metrics were compared.


              PROJECT EXECUTION COMPLETED


In [ ]:
# ==========================================
# STEP 13: CHECK FINAL ML DATASET
# ==========================================

print("Dataset Shape:")
print(ml_dataset.shape)

print("\nColumn Names:")
print(ml_dataset.columns.tolist())

print("\nFirst 10 Rows:")
display(ml_dataset.head(10))

print("\nBest Strategy Distribution:")
print(ml_dataset["Best_Strategy"].value_counts())

print("\nDataset Information:")
ml_dataset.info()

Dataset Shape:
(5000, 43)

Column Names:
['Workload_ID', 'Number_of_Processes', 'Average_Burst_Time', 'Minimum_Burst_Time', 'Maximum_Burst_Time', 'Burst_Time_Std', 'Burst_Time_Range', 'Average_Arrival_Time', 'Minimum_Arrival_Time', 'Maximum_Arrival_Time', 'Arrival_Time_Range', 'Arrival_Time_Std', 'Average_Priority', 'Minimum_Priority', 'Maximum_Priority', 'Priority_Std', 'CPU_Bound_Ratio', 'IO_Bound_Ratio', 'FCFS_Average_Waiting_Time', 'FCFS_Average_Turnaround_Time', 'FCFS_Average_Response_Time', 'FCFS_Throughput', 'FCFS_Context_Switches', 'FCFS_CPU_Utilization', 'SJF_Average_Waiting_Time', 'SJF_Average_Turnaround_Time', 'SJF_Average_Response_Time', 'SJF_Throughput', 'SJF_Context_Switches', 'SJF_CPU_Utilization', 'Priority_Average_Waiting_Time', 'Priority_Average_Turnaround_Time', 'Priority_Average_Response_Time', 'Priority_Throughput', 'Priority_Context_Switches', 'Priority_CPU_Utilization', 'RR_Average_Waiting_Time', 'RR_Average_Turnaround_Time', 'RR_Average_Response_Time', 'RR_Throu

,Workload_ID,Number_of_Processes,Average_Burst_Time,Minimum_Burst_Time,Maximum_Burst_Time,Burst_Time_Std,Burst_Time_Range,Average_Arrival_Time,Minimum_Arrival_Time,Maximum_Arrival_Time,...,Priority_Throughput,Priority_Context_Switches,Priority_CPU_Utilization,RR_Average_Waiting_Time,RR_Average_Turnaround_Time,RR_Average_Response_Time,RR_Throughput,RR_Context_Switches,RR_CPU_Utilization,Best_Strategy
0,1,4,3.750000,2,7,2.362908,5,3.500000,0,8,...,0.2667,3,100.00,1.75,5.50,1.25,0.2667,4,100.00,Round Robin
1,2,12,3.000000,1,6,1.906925,5,10.250000,0,20,...,0.3333,11,100.00,7.25,10.25,6.17,0.3333,14,100.00,SJF
2,3,7,4.571429,2,7,1.902379,5,10.714286,3,20,...,0.2188,6,100.00,8.57,13.14,4.86,0.2188,10,100.00,SJF
3,4,14,2.928571,1,6,1.730464,5,12.142857,2,20,...,0.3333,13,97.62,11.36,14.29,8.50,0.3333,16,97.62,SJF
4,5,8,18.875000,6,30,8.509449,24,1.750000,0,3,...,0.0530,7,100.00,92.12,111.00,12.25,0.0530,41,100.00,SJF
5,6,8,12.750000,2,30,10.067628,28,1.125000,0,3,...,0.0784,7,100.00,44.00,56.75,11.62,0.0784,26,100.00,SJF
6,7,12,13.583333,4,25,6.082140,21,9.750000,1,18,...,0.0736,11,100.00,91.00,104.58,18.58,0.0736,44,100.00,SJF
7,8,7,2.142857,1,5,1.676163,4,7.714286,0,13,...,0.3684,6,78.95,0.86,3.00,0.71,0.3684,7,78.95,FCFS
8,9,10,20.300000,16,30,5.437524,14,8.100000,1,16,...,0.0493,9,100.00,134.50,154.80,16.10,0.0493,53,100.00,SJF
9,10,10,4.200000,2,7,1.751190,5,11.200000,2,20,...,0.2381,9,100.00,11.00,15.20,8.00,0.2381,13,100.00,SJF



Best Strategy Distribution:
Best_Strategy
SJF            4562
FCFS            243
Priority        136
Round Robin      59
Name: count, dtype: int64

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 43 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Workload_ID                       5000 non-null   int64  
 1   Number_of_Processes               5000 non-null   int64  
 2   Average_Burst_Time                5000 non-null   float64
 3   Minimum_Burst_Time                5000 non-null   int64  
 4   Maximum_Burst_Time                5000 non-null   int64  
 5   Burst_Time_Std                    5000 non-null   float64
 6   Burst_Time_Range                  5000 non-null   int64  
 7   Average_Arrival_Time              5000 non-null   float64
 8   Minimum_Arrival_Time              5000 non-null   int64  
 9   Maximum_Arrival_Time    

In [ ]:
# ==========================================
# STEP 14: DOWNLOAD FINAL DATASET
# ==========================================
from google.colab import files

files.download("scheduling_ml_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>